
# Architecture for Freight Invoice Processing Solution

The architecture for the freight invoice processing solution is shown below. The process begins with a Freight Invoice PDF received from carriers via email:

## Processing Steps

**Step 1:** Carrier sends freight invoice via email, which is polled by a scheduled Lambda function using Microsoft Graph API and stored in an S3 bucket (invoice-repository).

**Step 2:** An S3 Object Created event is matched by a configured EventBridge Rule when a new invoice PDF is uploaded.

**Step 3:** EventBridge rule triggers an AWS Lambda function (InvokeBDALambda).

**Step 4:** The Lambda function calls the InvokeDataAutomationAsync job in Bedrock Data Automation (BDA) along with a custom blueprint configured for Phillips 66 freight invoices.

**Step 5:** BDA uses the provided custom blueprint for freight invoice extraction to extract structured data from the invoice PDF including shipper, consignee, lane information, freight charges, fuel surcharges, and accessorial charges. BDA stores extracted invoice data in the S3 bucket specified in the API call.

**Step 6:** BDA sends a Job Completion event to EventBridge that includes the job status and the S3 URI of the response and metadata.

**Step 7:** EventBridge rule triggers an AWS Lambda function (ProcessBDAResultsLambda).

**Step 8:** The Lambda function uses the BDA job response and metadata to fetch the extracted invoice data from S3.

**Step 9:** The Lambda function stores the extracted invoice data in GVP database and validates required fields.

## Future Enhancements (Phase 1)

**Step 10:** Bedrock Agent will be invoked to process invoice disputes by analyzing discrepancies between extracted data and contract terms.

**Step 11:** Bedrock Agent uses configured agent actions (fulfilled by Lambda functions) to perform tasks required during dispute resolution process, including composing and interpreting carrier communication emails.

**Step 12:** Bedrock Agent uses agent action, implemented using Lambda function, to query relevant invoice data stored in GVP to validate identify discrepancies.

**Step 13:** Bedrock Agent queries Bedrock Knowledge Base to gather contract details, rate agreements, and accessorial charge policies relevant to the invoice validation.

**Step 14:** Bedrock Agent creates a final validation report detailing any discrepancies, recommended adjustments, and approval/rejection decision, then stores the report in S3 for user review.


In [43]:
import boto3
import json
# Initialize Bedrock Data Automation client
bda_client = boto3.client(
    'bedrock-data-automation',
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

# Initialize Bedrock Data Automation Runtime client
bda_runtime_client = boto3.client(
    'bedrock-data-automation-runtime',
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

# Initialize S3 client
s3_client = boto3.client(
    's3',
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

# Create STS client
sts_client = boto3.client(
    "sts",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

# Get AWS Account ID
account_id = sts_client.get_caller_identity()["Account"]
current_region = AWS_REGION

print("AWS Account ID:", account_id)

AWS Account ID: 721937028010


In [3]:
bda_s3_input_location = f's3://devgvpbucket1/Invoices'
bda_s3_output_location = f's3://devgvpbucket1/bda-output'
local_file_name = "CPKC.pdf"
document_s3_uri = f'{bda_s3_input_location}/{local_file_name}'

In [21]:
from urllib.parse import urlparse
from IPython.display import display, HTML
import ipywidgets as widgets
import time


def read_s3_object(s3_uri):
    """
    Reads an S3 object and returns its content as a string.
    """
    parsed_uri = urlparse(s3_uri)
    bucket_name = parsed_uri.netloc
    object_key = parsed_uri.path.lstrip('/')
    
    try:
        response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
        content = response['Body'].read().decode('utf-8')
        return content
    except Exception as e:
        print(f"Error reading S3 object {s3_uri}: {e}")
        return None



def json_to_html(json_obj, indent=0):
    result = []
    if isinstance(json_obj, dict):
        result.append('<table class="json-object">')
        for key, value in json_obj.items():
            result.append('<tr>')
            result.append(f'<td class="key">{key}</td>')
            result.append('<td class="value">')
            result.append(json_to_html(value, indent + 1))
            result.append('</td>')
            result.append('</tr>')
        result.append('</table>')
    elif isinstance(json_obj, list):
        result.append('<table class="json-array">')
        for i, item in enumerate(json_obj):
            result.append('<tr>')
            result.append(f'<td class="key">{i}</td>')
            result.append('<td class="value">')
            result.append(json_to_html(item, indent + 1))
            result.append('</td>')
            result.append('</tr>')
        result.append('</table>')
    elif isinstance(json_obj, (str, int, float, bool)) or json_obj is None:
        if isinstance(json_obj, str):
            result.append(f'<span class="string">"{json_obj}"</span>')
        elif isinstance(json_obj, bool):
            result.append(f'<span class="boolean">{str(json_obj).lower()}</span>')
        elif json_obj is None:
            result.append('<span class="null">null</span>')
        else:
            result.append(f'<span class="number">{json_obj}</span>')
    return ''.join(result)


def display_json(json_data, title="JSON"):
    html_content = f"""
    <div class="json-container">
        <h3 class="json-title">{title}</h3>
        <div class="json-viewer">
            {json_to_html(json_data)}
        </div>
    </div>
    <style>
        .json-container {{
            margin-bottom: 20px;
        }}
        .json-title {{
            font-family: sans-serif;
            font-size: 18px;
            font-weight: bold;
            margin-bottom: 10px;
            color: #333;
        }}
        .json-viewer {{
            font-family: monospace;
            font-size: 14px;
            line-height: 1.5;
            background-color: #f8f8f8;
            border: 1px solid #ddd;
            border-radius: 4px;
            padding: 10px;
            max-height: 500px;
            overflow: auto;
        }}
        .json-object, .json-array {{
            border-collapse: collapse;
            margin-left: 20px;
        }}
        .key {{
            color: #881391;
            vertical-align: top;
            padding-right: 10px;
        }}
        .value {{
            padding-left: 10px;
        }}
        .string {{ color: #1a1aa6; }}
        .number {{ color: #116644; }}
        .boolean {{ color: #ff8c00; }}
        .null {{ color: #808080; }}
    </style>
    """
    display(widgets.HTML(html_content))


def wait_for_job_to_complete(invocationArn, timeout=600, poll_interval=1):
    """
    Polls the Bedrock Data Automation Runtime invocation until it reaches a terminal state.
    """
    start_time = time.time()
    while time.time() - start_time < timeout:
        response = bda_runtime_client.get_data_automation_status(invocationArn=invocationArn)
        status = response.get("status")
        print(f"Current status: {status}")
        
        if status in ['Success', 'Failed', 'Cancelled']:
            return response
        
        time.sleep(poll_interval)
    
    raise TimeoutError("Job did not complete within the specified timeout")


In [8]:
# create blueprint using Boto3
blueprints = [
    {
        "name": 'invoice_schema',
        "description": 'Blueprint for invoice',
        "type": 'DOCUMENT',
        "stage": 'LIVE',
        "schema_path": 'data/blueprints/invoice_schema.json'
    }]

In [9]:
from utils import helper_functions,display_functions

In [12]:
blueprint_arns = []
for blueprint in blueprints:
    with open(blueprint['schema_path']) as f:
        blueprint_schema = json.load(f)
        blueprint_arn = helper_functions.create_or_update_blueprint(
            bda_client, 
            blueprint['name'], 
            blueprint['description'], 
            blueprint['type'],
            blueprint['stage'],
            blueprint_schema
        )
        blueprint_arns += [blueprint_arn]

Found existing blueprint with name=invoice_schema, updating Stage and Schema


In [11]:
import json
with open("data/blueprints/invoice_schema.json", "r") as f:
    schema_data = json.load(f)
print(json.dumps(schema_data, indent=2))

{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "description": "Railway Invoice Schema",
  "class": "Railway Invoice",
  "type": "object",
  "definitions": {},
  "properties": {
    "Company_Code": {
      "type": "string",
      "inferenceType": "explicit",
      "instruction": "Extract the company code identifier from the invoice"
    },
    "Invoice_Number": {
      "type": "string",
      "inferenceType": "explicit",
      "instruction": "Extract Invoice number from provided invoice"
    },
    "Invoice_Received_Date": {
      "type": "string",
      "inferenceType": "explicit",
      "instruction": "Extract the date the invoice was received"
    },
    "Invoice_Type": {
      "type": "string",
      "inferenceType": "explicit",
      "instruction": "Extract the type of invoice"
    },
    "Invoice_Date": {
      "type": "string",
      "inferenceType": "explicit",
      "instruction": "Extract the invoice date"
    },
    "Party_Name": {
      "type": "string",
      "

In [33]:
bda_project_name = 'Freight_Audit_Agent'
bda_project_stage = 'LIVE'
standard_output_configuration = {
    'document': {
        'extraction': {
            'granularity': {'types': ['DOCUMENT']},
            'boundingBox': {'state': 'DISABLED'}
        },
        'generativeField': {'state': 'ENABLED'},
        'outputFormat': {
            'textFormat': {'types': ['MARKDOWN']},
            'additionalFileFormat': {'state': 'ENABLED'}
        }
    }
    }


custom_output_configuration = {
    "blueprints": [    ]
}
custom_output_configuration['blueprints'] += [
    {
        'blueprintArn': blueprint_arn,
        'blueprintStage': 'LIVE'
    } for blueprint_arn in blueprint_arns
]

override_configuration={'document': {'splitter': {'state': 'ENABLED'}}}
print(custom_output_configuration["blueprints"])

[{'blueprintArn': 'arn:aws:bedrock:us-east-1:721937028010:blueprint/31f568178014', 'blueprintStage': 'LIVE'}]


In [34]:
list_project_response = bda_client.list_data_automation_projects(
    projectStageFilter=bda_project_stage)

project = next((project for project in list_project_response['projects']
               if project['projectName'] == bda_project_name), None)

if not project:
    response = bda_client.create_data_automation_project(
        projectName=bda_project_name,
        projectDescription='Invoice document processing combining blueprints with data projects',
        projectStage=bda_project_stage,
        standardOutputConfiguration=standard_output_configuration,
        customOutputConfiguration=custom_output_configuration,
        overrideConfiguration=override_configuration
    )
else:
    response = bda_client.update_data_automation_project(
        projectArn=project['projectArn'],
        standardOutputConfiguration=standard_output_configuration,
        customOutputConfiguration=custom_output_configuration,
        overrideConfiguration=override_configuration
    )

project_arn = response['projectArn']


In [35]:
status_response = helper_functions.wait_for_completion(
            client=bda_client,
            get_status_function=bda_client.get_data_automation_project,
            status_kwargs={'projectArn': project_arn},
            completion_states=['COMPLETED'],
            error_states=['FAILED'],
            status_path_in_response='project.status',
            max_iterations=15,
            delay=30
)

Operation completed successfully with status: COMPLETED


In [36]:
project_arn

'arn:aws:bedrock:us-east-1:721937028010:data-automation-project/e202452398ba'

In [16]:
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': document_s3_uri
    },
    outputConfiguration={
        's3Uri': bda_s3_output_location
    },
    dataAutomationConfiguration={
        'dataAutomationProjectArn': project_arn,
        'stage': 'LIVE'
    }, 
    dataAutomationProfileArn = f'arn:aws:bedrock:{current_region}:{account_id}:data-automation-profile/us.data-automation-v1'
)

invocationArn = response['invocationArn']
print(f'Invoked data automation job with invocation arn {invocationArn}')

Invoked data automation job with invocation arn arn:aws:bedrock:us-east-1:721937028010:data-automation-invocation/c24db13c-ab8c-4f0a-a6f6-11164f95da57


In [45]:
region =  'us-east-1'
account_id = '721937028010'
project_id = 'e202452398ba'

# bda_client = boto3.client('bedrock-data-automation', region_name=region)
project_arn = f'arn:aws:bedrock:{region}:{account_id}:data-automation-project/{project_id}'

response = bda_client.get_data_automation_project(
    projectArn=project_arn
)

print("Current project configuration:")
print(response)
# print(json.dumps(response['dataAutomationProject']['standardOutputConfiguration'], indent=2))

Current project configuration:
{'ResponseMetadata': {'RequestId': '57ba0f79-15e7-4d09-9822-d9007b79ea41', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 02 Sep 2025 15:19:21 GMT', 'content-type': 'application/json', 'content-length': '1329', 'connection': 'keep-alive', 'x-amzn-requestid': '57ba0f79-15e7-4d09-9822-d9007b79ea41'}, 'RetryAttempts': 0}, 'project': {'projectArn': 'arn:aws:bedrock:us-east-1:721937028010:data-automation-project/e202452398ba', 'creationTime': datetime.datetime(2025, 8, 14, 15, 9, 43, 220000, tzinfo=tzutc()), 'lastModifiedTime': datetime.datetime(2025, 9, 2, 15, 5, 3, 856000, tzinfo=tzutc()), 'projectName': 'Freight_Audit_Agent', 'projectStage': 'LIVE', 'standardOutputConfiguration': {'document': {'extraction': {'granularity': {'types': ['DOCUMENT']}, 'boundingBox': {'state': 'DISABLED'}}, 'generativeField': {'state': 'ENABLED'}, 'outputFormat': {'textFormat': {'types': ['MARKDOWN']}, 'additionalFileFormat': {'state': 'ENABLED'}}}, 'image': {'extraction':

In [46]:
print(json.dumps(response['project']['standardOutputConfiguration'], indent=2))

{
  "document": {
    "extraction": {
      "granularity": {
        "types": [
          "DOCUMENT"
        ]
      },
      "boundingBox": {
        "state": "DISABLED"
      }
    },
    "generativeField": {
      "state": "ENABLED"
    },
    "outputFormat": {
      "textFormat": {
        "types": [
          "MARKDOWN"
        ]
      },
      "additionalFileFormat": {
        "state": "ENABLED"
      }
    }
  },
  "image": {
    "extraction": {
      "category": {
        "state": "ENABLED",
        "types": [
          "TEXT_DETECTION"
        ]
      },
      "boundingBox": {
        "state": "ENABLED"
      }
    },
    "generativeField": {
      "state": "ENABLED",
      "types": [
        "IMAGE_SUMMARY"
      ]
    }
  },
  "video": {
    "extraction": {
      "category": {
        "state": "ENABLED",
        "types": [
          "TEXT_DETECTION"
        ]
      },
      "boundingBox": {
        "state": "ENABLED"
      }
    },
    "generativeField": {
      "state": "EN

In [47]:
print(json.dumps(response['project']['customOutputConfiguration'], indent=2))

{
  "blueprints": [
    {
      "blueprintArn": "arn:aws:bedrock:us-east-1:721937028010:blueprint/31f568178014",
      "blueprintStage": "LIVE"
    }
  ]
}


In [17]:
status_response = helper_functions.wait_for_completion(
            client=bda_client,
            get_status_function=bda_runtime_client.get_data_automation_status,
            status_kwargs={'invocationArn': invocationArn},
            completion_states=['Success'],
            error_states=['ClientError', 'ServiceError'],
            status_path_in_response='status',
            max_iterations=15,
            delay=30
)
if status_response['status'] == 'Success':
    job_metadata_s3_location = status_response['outputConfiguration']['s3Uri']
else:
    raise Exception(f'Invocation Job Error, error_type={status_response["error_type"]},error_message={status_response["error_message"]}')

Current status: InProgress. Waiting...
Operation completed successfully with status: Success


In [18]:
# Initialize S3 client
s3_client = boto3.client(
    's3',
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)


In [19]:
import pandas as pd

In [23]:
job_metadata = json.loads(read_s3_object(job_metadata_s3_location))

job_metadata_table = pd.DataFrame(job_metadata['output_metadata'][0]['segment_metadata']).fillna('')
job_metadata_table.index.name='Segment Index'
# job_metadata_json = JSON(job_metadata, root="job_metadata", expanded=True)

job_metadata_json = json.dumps(job_metadata, indent=2, default=str)

# Display the widget
display_functions.display_multiple(
    [display_functions.get_view(job_metadata_table), display_functions.get_view(job_metadata_json)], 
    ["Table View", "Raw JSON"])

In [24]:
status_response = wait_for_job_to_complete(invocationArn=response["invocationArn"])
status_response

Current status: Success


{'status': 'Success',
 'outputConfiguration': {'s3Uri': 's3://devgvpbucket1/bda-output/c24db13c-ab8c-4f0a-a6f6-11164f95da57/job_metadata.json'},
 'ResponseMetadata': {'RequestId': 'fc522236-0a66-40f6-a6e6-96e479e931c2',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 02 Sep 2025 14:22:33 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '139',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'fc522236-0a66-40f6-a6e6-96e479e931c2'},
  'RetryAttempts': 0}}

In [27]:
asset_id = 0
segments_metadata = next(item["segment_metadata"]
                                for item in job_metadata["output_metadata"] 
                                if item['asset_id'] == asset_id)

standard_outputs = [
    json.loads(read_s3_object(segment_metadata.get('standard_output_path')))for segment_metadata in segments_metadata]
custom_outputs = [json.loads(read_s3_object(segment_metadata.get('custom_output_path'))) if segment_metadata.get('custom_output_status') == 'MATCH' else None for segment_metadata in segments_metadata]

In [80]:
def read_s3_object(s3_uri,s3_client):
    # Parse the S3 URI
    parsed_uri = urlparse(s3_uri)
    bucket_name = parsed_uri.netloc
    object_key = parsed_uri.path.lstrip('/')
    
    # Use provided S3 client or create a new one
    if s3_client is None:
        s3_client = boto3.client('s3')
    
    try:
        # Get the object from S3
        response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
        
        # Read the content of the object
        content = response['Body'].read().decode('utf-8')
        return content
    except Exception as e:
        print(f"Error reading S3 object: {e}")
        return None


In [28]:

# custom_outputs_json = JSON(custom_outputs, root="custom_outputs", expanded=False)
custom_outputs_json = json.dumps(custom_outputs, indent=2, default=str)
custom_outputs_table = pd.DataFrame(helper_functions.get_summaries(custom_outputs)).fillna('')

display_functions.display_multiple(
    [
        display_functions.get_view(custom_outputs_table.style.hide(axis='index')),
        display_functions.get_view(custom_outputs_json)
    ], 
    ["Table View", "Raw JSON"])

In [29]:

print(json.dumps(json.loads(custom_outputs_json), indent=2))

[
  {
    "matched_blueprint": {
      "arn": "arn:aws:bedrock:us-east-1:721937028010:blueprint/31f568178014",
      "name": "invoice_schema",
      "confidence": 1
    },
    "document_class": {
      "type": "Railway Invoice"
    },
    "split_document": {
      "page_indices": [
        0
      ]
    },
    "inference_result": {
      "Charge_1_Cost_Center": "",
      "Charge_2_GLAccount": "",
      "Charge_1_Amount": 500,
      "Ship_Date": "2025/06/03",
      "Charge_2_Amount": "",
      "Charge_3_Cost_Center": "",
      "Charge_1_Quantity": 1,
      "Charge_4_Cost_Center": "",
      "Charge_4_GLAccount": "",
      "Charge_5_Cost_Center": "",
      "Invoice_Received_Date": "2025/06/04",
      "Invoice_Date": "2025/06/04",
      "Party_Name": "ARDENT MILLS LLC",
      "Charge_3_Quantity": "",
      "Address_1": "15407 MCGINTY ROAD W",
      "Charge_5_Amount": "",
      "Charge_4_Amount": "",
      "Charge_3_Amount": "",
      "BOL_Number": "",
      "Company_Code": "CPKC",
      "C